# Query-to-SubData Selection trên Google Colab

Notebook triển khai pipeline lấy cảm hứng từ **LLM-Free Visual Selection của LightSTAR**: tạo biểu diễn nhẹ, route query theo corpus → document → page, rồi chỉ full parse và full embed SubData đã chọn.

> Khuyến nghị: **Runtime → Change runtime type → T4 GPU**. Light Preparation không dùng LLM và không full parse toàn bộ corpus.

## 1. Clone mã nguồn và cài thư viện

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/ManhTanTran/data-discovery.git"
REPO_DIR = Path("/content/data-discovery")

if not (REPO_DIR / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_DIR}[ml]"],
    check=True,
)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print("Đã cài đặt Data Discovery thành công.")

## 2. Mount Google Drive và tìm corpus

Nếu thư mục là **Shared with me**, hãy tạo shortcut của `vidore_v3_industrial` vào **My Drive** trước khi chạy.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# Có thể điền trực tiếp nếu notebook không tự tìm thấy.
DATA_DIR_OVERRIDE = ""
my_drive = Path("/content/drive/MyDrive")

if DATA_DIR_OVERRIDE:
    data_dir = Path(DATA_DIR_OVERRIDE)
else:
    candidates = [
        my_drive / "vidore_v3_industrial" / "pdfs",
        my_drive / "iSE_DE" / "vidore_v3" / "vidore_v3_industrial" / "pdfs",
    ]
    data_dir = next((path for path in candidates if path.exists()), None)
    if data_dir is None:
        matches = list(my_drive.glob("**/vidore_v3_industrial/pdfs"))
        data_dir = matches[0] if matches else None

if data_dir is None or not data_dir.exists():
    raise FileNotFoundError(
        "Không tìm thấy thư mục pdfs. Hãy tạo shortcut vào My Drive hoặc đặt DATA_DIR_OVERRIDE."
    )
print(f"Dữ liệu được chọn: {data_dir}")
print(f"Số PDF: {len(list(data_dir.glob('*.pdf')))}")

## 3. Cấu hình selection

`alpha`, `beta`, `gamma` lần lượt là trọng số lexical, semantic và metadata. Exploration 5% lấy ngẫu nhiên một phần candidate bị loại để giảm false negative.

In [ ]:
import torch
from src.data_discovery import DiscoveryConfig

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
LIGHT_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

config = DiscoveryConfig(
    top_k_corpora=1,
    top_k_documents=8,
    top_k_pages=12,
    late_interaction_top_k=3,
    selection_threshold=0.15,
    alpha=0.25,
    beta=0.60,
    gamma=0.15,
    exploration_rate=0.05,
    max_preview_chars=1200,
    max_preview_segments_per_document=64,
    create_pdf_thumbnails=False,
    ann_backend="faiss",
    chunk_size_words=180,
    chunk_overlap_words=30,
    final_top_k=5,
)
print(f"Thiết bị embedding: {DEVICE}")
print(f"Model nhẹ: {LIGHT_MODEL}")

## 4. Light Preparation

PDF chỉ được đọc text preview giới hạn theo trang. Với HTML/TXT/DOCX, module lấy title, heading và preview segment; không gọi LLM.

In [ ]:
from src.data_discovery import LightPreparer

# Có thể thêm nhiều corpus: {"manuals": path_a, "reports": path_b}.
corpus_roots = {"vidore_industrial": data_dir}
manifest = LightPreparer(config).prepare(corpus_roots)

print(f"Số corpus: {len(manifest.corpora)}")
print(f"Số document: {len(manifest.documents)}")
print(f"Số preview/page segment: {len(manifest.segments)}")
print(f"Thời gian Light Preparation: {manifest.preparation_latency_ms:.1f} ms")

## 5. Tạo Light Index

Index gồm single-vector ở mỗi cấp và multi-vector từ các preview segment. FAISS dùng cho candidate retrieval; late interaction dùng để rerank document/page.

In [ ]:
from src.data_discovery import LightIndex, SentenceTransformerEmbedder

light_embedder = SentenceTransformerEmbedder(
    LIGHT_MODEL, device=DEVICE, batch_size=64
)
light_index = LightIndex(
    manifest, light_embedder, ann_backend=config.ann_backend
)
print("Backend thực tế:", light_index.backend_used)

## 6. Query → Top-K SubData

In [ ]:
import pandas as pd
from IPython.display import display
from src.data_discovery import QueryRouter

QUERY = "Which page describes the system architecture and its main components?"
selection = QueryRouter(light_index, config).select(QUERY)

def selection_frame(items):
    return pd.DataFrame([
        {
            "item_id": item.item_id,
            "corpus_id": item.corpus_id,
            "document_id": item.document_id,
            "page_id": item.page_id,
            "score": round(item.score, 4),
            "lexical": round(item.lexical_score, 4),
            "semantic": round(item.semantic_score, 4),
            "metadata": round(item.metadata_score, 4),
            "exploration": item.exploration,
        } for item in items
    ])

print("Corpus được chọn:")
display(selection_frame(selection.corpora))
print("Document được chọn:")
display(selection_frame(selection.documents))
print("Page/segment được chọn:")
display(selection_frame(selection.pages))
print("Kế hoạch xử lý đầy đủ:")
display(pd.DataFrame(selection.processing_plan))
print("Số item bị loại:", selection.eliminated)
print("Latency selection (ms):", selection.latency_ms)
print(f"Giảm parsing ước tính: {selection.cost.parsing_cost_reduction:.2%}")
print(f"Giảm embedding ước tính: {selection.cost.embedding_cost_reduction:.2%}")

## 7. Full Processing có chọn lọc

Cell này chỉ mở và full parse các page/document xuất hiện trong `processing_plan`; tài liệu không vượt candidate threshold sẽ không được full embed.

In [ ]:
from src.data_discovery import FullProcessor

# Có thể thay bằng model mạnh hơn cho bước cuối; mặc định tái sử dụng model nhẹ để tiết kiệm VRAM.
full_result = FullProcessor(manifest, light_embedder, config).process(selection)

hits_df = pd.DataFrame([
    {
        "document_id": hit.document_id,
        "page_id": hit.page_id,
        "score": round(hit.score, 4),
        "text": hit.text[:500],
    } for hit in full_result.hits
])
display(hits_df)
print(f"Đã full parse {full_result.parsed_units} unit, tạo {full_result.chunks} chunk.")
print("Document thực sự được xử lý:", full_result.processed_document_ids)
print("Page thực sự được xử lý:", full_result.processed_page_ids)
print("Latency full processing (ms):", full_result.latency_ms)

## 8. Đánh giá Recall@K (tùy chọn)

Điền ID ground truth từ qrels/dataset. Nếu để trống, cell vẫn báo candidate/cost reduction nhưng Recall@K mặc định bằng 1 do chưa có nhãn liên quan.

In [ ]:
from src.data_discovery.evaluation import GroundTruth, evaluate_selection

RELEVANT_CORPUS_IDS = set()
RELEVANT_DOCUMENT_IDS = set()
RELEVANT_PAGE_IDS = set()

report = evaluate_selection(
    selection,
    manifest,
    GroundTruth(RELEVANT_CORPUS_IDS, RELEVANT_DOCUMENT_IDS, RELEVANT_PAGE_IDS),
    full_result=full_result,
    top_k_corpora=config.top_k_corpora,
    top_k_documents=config.top_k_documents,
    top_k_pages=config.top_k_pages,
)
display(pd.DataFrame([report.to_dict()]))

## 9. Lưu kết quả vào Google Drive

In [ ]:
from dataclasses import asdict
from datetime import datetime
import json

output_dir = my_drive / "data_discovery_outputs" / datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir.mkdir(parents=True, exist_ok=True)
payloads = {
    "selection.json": selection.to_dict(),
    "full_result.json": full_result.to_dict(),
    "evaluation.json": report.to_dict(),
    "config.json": asdict(config),
}
for filename, payload in payloads.items():
    (output_dir / filename).write_text(
        json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8"
    )
print(f"Đã lưu kết quả tại: {output_dir}")

## Liên hệ với LightSTAR

- **Tương ứng LightSTAR:** preview ảnh/text độ phân giải thấp, model nhẹ, truy hồi candidate và late interaction mà không cần LLM.
- **Phần mở rộng:** routing phân cấp corpus → document → page; hybrid lexical/semantic/metadata; hỗ trợ HTML, DOCX, TXT; random exploration 5%; ước tính chi phí và selective full processing.
- **Ranh giới quan trọng:** LightSTAR là cảm hứng thiết kế selection. Pipeline này không tuyên bố là bản tái tạo chính thức hoặc đầy đủ của paper.